In [ ]:
import spacy
from spacy.language import Language
from spacy.tokens import Doc
from spacy.matcher import PhraseMatcher, Matcher
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import torch
from typing import Dict, List, Set, Tuple
import numpy as np
from typing import List, Dict, Tuple
from spacy.tokens import Token, Doc
# Attempt to use MPS (Metal Performance Shaders) if available
torch.device('mps')  # Uncomment if you are on an Apple Silicon Mac
spacy.require_gpu()
nlp = spacy.load("en_core_web_trf")
# Configure patterns and pandas display
pattern = r'ICLE\-\w+\-\w+\-\d+\.\d+'
pattern = r'[^\w\s]'
pd.set_option('display.max_colwidth', None)
# Load and preprocess data
meta = pd.read_csv('/Users/fatihbozdag/Documents/Studies/AI Library/Metadata/metadata_with_text.csv')  # Update with your path
text = pd.read_csv('/Users/fatihbozdag/Documents/Studies/AI Library/Metadata/metadata_with_text.csv')  # Update with your path
text['text_field'] = text['text_field'].apply(lambda x: re.sub(pattern, '', x).replace('\n', ''))
text['text_field'] = text['text_field'].apply(lambda x: re.sub(pattern, '', x).replace('\n', ''))
text['text_field'] = text['text_field'].str.lower()
meta_x = meta.to_dict('records')
text_only = text['text_field'].values.tolist()
icle = list(zip(text_only, meta_x))

In [ ]:
# Define comprehensive metadiscourse marker lists based on Hyland (2005)
INTERACTIVE_MARKERS = {
    # Code glosses
    "code_glosses": [
        "in other words", "that is", "i.e.", "that is to say", "this means", "in simple terms",
        "put simply", "to put it simply", "namely", "for example", "for instance", "such as", 
        "e.g.", "specifically", "particularly", "in fact", "indeed", "actually", "called", 
        "defined as", "referred to as", "including", "included", "especially", "notably"
    ],
    
    # Endophoric markers
    "endophoric_markers": [
        "in chapter", "in section", "in part", "in figure", "in table", "figure", "table", 
        "above", "below", "earlier", "previously", "as noted above", "as mentioned earlier", 
        "see", "refer to", "page", "the following", "as follows", "aforementioned"
    ],
    
    # Evidentials
    "evidentials": [
        "according to", "cited", "quoted", "states that", "argues that", "notes that", 
        "suggests that", "reports that", "found that", "observed that", "concluded that", 
        "in the literature", "previous research", "research shows", "studies indicate"
    ],
    
    # Frame markers
    "frame_markers": [
        # Sequencers
        "first", "firstly", "second", "secondly", "third", "thirdly", "fourth", "finally", 
        "lastly", "to begin with", "to start with", "next", "then", "subsequently",
        # Stage labels
        "in conclusion", "to conclude", "to summarize", "in summary", "in brief", "all in all", 
        "on the whole", "so far", "at this point", "overall",
        # Goal announcements
        "aim", "purpose", "goal", "objective", "focus", "seek to", "intend to", 
        # Topic shifters
        "with regard to", "concerning", "regarding", "turning to", "moving on to", "back to"
    ],
    
    # Transition markers
    "transition_markers": [
        # Additive
        "moreover", "furthermore", "in addition", "additionally", "besides", "similarly", 
        "likewise", "equally", "also", 
        # Causal
        "therefore", "thus", "consequently", "hence", "as a result", "because", "since", 
        "due to", "owing to", "so", 
        # Adversative
        "however", "nevertheless", "nonetheless", "but", "yet", "though", "although", "even though", 
        "despite", "in spite of", "in contrast", "on the other hand", "conversely", 
        # Temporal
        "meanwhile", "simultaneously", "subsequently", "previously", "after", "before", 
        "then", "later", "formerly", "eventually"
    ]
}

In [ ]:
# Define comprehensive metadiscourse marker lists based on Hyland (2005)
INTERACTIVE_MARKERS = {
    # Code glosses
    "code_glosses": [
        "in other words", "that is", "i.e.", "that is to say", "this means", "in simple terms",
        "put simply", "to put it simply", "namely", "for example", "for instance", "such as", 
        "e.g.", "specifically", "particularly", "in fact", "indeed", "actually", "called", 
        "defined as", "referred to as", "including", "included", "especially", "notably"
    ],
    
    # Endophoric markers
    "endophoric_markers": [
        "in chapter", "in section", "in part", "in figure", "in table", "figure", "table", 
        "above", "below", "earlier", "previously", "as noted above", "as mentioned earlier", 
        "see", "refer to", "page", "the following", "as follows", "aforementioned"
    ],
    
    # Evidentials
    "evidentials": [
        "according to", "cited", "quoted", "states that", "argues that", "notes that", 
        "suggests that", "reports that", "found that", "observed that", "concluded that", 
        "in the literature", "previous research", "research shows", "studies indicate"
    ],
    
    # Frame markers
    "frame_markers": [
        # Sequencers
        "first", "firstly", "second", "secondly", "third", "thirdly", "fourth", "finally", 
        "lastly", "to begin with", "to start with", "next", "then", "subsequently",
        # Stage labels
        "in conclusion", "to conclude", "to summarize", "in summary", "in brief", "all in all", 
        "on the whole", "so far", "at this point", "overall",
        # Goal announcements
        "aim", "purpose", "goal", "objective", "focus", "seek to", "intend to", 
        # Topic shifters
        "with regard to", "concerning", "regarding", "turning to", "moving on to", "back to"
    ],
    
    # Transition markers
    "transition_markers": [
        # Additive
        "moreover", "furthermore", "in addition", "additionally", "besides", "similarly", 
        "likewise", "equally", "also", 
        # Causal
        "therefore", "thus", "consequently", "hence", "as a result", "because", "since", 
        "due to", "owing to", "so", 
        # Adversative
        "however", "nevertheless", "nonetheless", "but", "yet", "though", "although", "even though", 
        "despite", "in spite of", "in contrast", "on the other hand", "conversely", 
        # Temporal
        "meanwhile", "simultaneously", "subsequently", "previously", "after", "before", 
        "then", "later", "formerly", "eventually"
    ]
}

In [ ]:
INTERACTIONAL_MARKERS = {
    # Attitude markers
    "attitude_markers": [
        "unfortunately", "fortunately", "surprisingly", "remarkably", "interestingly", 
        "hopefully", "importantly", "significantly", "correctly", "appropriately", "agree", 
        "prefer", "disagree", "dramatic", "unexpected", "desirable", "disappointing", "alarming",
        "it is surprising that", "it is important that", "it is significant that"
    ],
    
    # Self-mention
    "self_mention": [
        "i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves", 
        "the author", "the authors", "the researcher", "the researchers", "this author"
    ],
    
    # Engagement markers
    "engagement_markers": [
        "you", "your", "yours", "yourself", "consider", "note", "imagine", "think about", 
        "let us", "let's", "see", "must", "should", "need to", "have to", "ought to", 
        "what about", "how about", "by the way", "the reader", "readers"
    ],
    
    # Hedges
    "hedges": [
        "may", "might", "could", "would", "perhaps", "possibly", "probably", "maybe", "likely", 
        "seemingly", "apparently", "approximately", "about", "roughly", "suggest", "assume", 
        "believe", "think", "appear", "seem", "indicate", "suspect", "suppose", "estimate", 
        "in my opinion", "from my perspective", "to my knowledge", "generally", "usually", 
        "sometimes", "often", "in most cases", "to some extent", "sort of", "kind of"
    ],
    
    # Boosters
    "boosters": [
        "clearly", "obviously", "certainly", "definitely", "undoubtedly", "undeniably", 
        "demonstrate", "prove", "show", "establish", "confirm", "find", "reveal", "must", 
        "will", "beyond doubt", "without doubt", "in fact", "indeed", "actually", "always", 
        "never", "absolutely", "completely", "entirely", "truly", "really", 
        "it is clear that", "we found that", "we proved that"
    ]
}

# Context rules to filter false positives
CONTEXT_PATTERNS = {
    "hedges": [
        # Modal verbs functioning as hedges (check they're not followed by nouns)
        [{"LOWER": {"IN": ["may", "might", "could", "would"]}}, {"TAG": {"NOT_IN": ["NN", "NNP"]}, "OP": "+"}, {"TAG": "VB"}],
        # "I think/believe" pattern
        [{"LOWER": {"IN": ["i", "we"]}}, {"LOWER": {"IN": ["think", "believe", "assume", "suppose"]}}],
        # "It appears/seems" pattern
        [{"LOWER": "it"}, {"LOWER": {"IN": ["appears", "seems", "looks"]}}],
        # "likely to" - epistemic rather than similarity
        [{"LOWER": "likely"}, {"LOWER": "to"}],
    ],
    
    "frame_markers": [
        # Sequencers followed by a comma often indicate discourse organization
        [{"LOWER": {"IN": ["first", "second", "third", "finally", "lastly"]}}, {"IS_PUNCT": True, "LOWER": ","}],
        # Goal statements
        [{"LOWER": {"IN": ["i", "we", "this", "the"]}}, {"LOWER": {"IN": ["aim", "intend", "focus", "purpose"]}}],
        # Topic shifters
        [{"LOWER": {"IN": ["turning", "moving"]}}, {"LOWER": "to"}],
    ],
    
    "transition_markers": [
        # Transitions at sentence start
        [{"IS_SENT_START": True}, {"LOWER": {"IN": ["however", "nevertheless", "thus", "therefore"]}}],
        # "Because of" causal pattern
        [{"LOWER": "because"}, {"LOWER": "of"}],
        # "As a result" pattern
        [{"LOWER": "as"}, {"LOWER": "a"}, {"LOWER": "result"}],
    ],
    
    "evidentials": [
        # "According to" pattern
        [{"LOWER": "according"}, {"LOWER": "to"}],
        # "X states/argues that" pattern
        [{"POS": "PROPN"}, {"LOWER": {"IN": ["states", "argues", "claims", "notes"]}}, {"LOWER": "that"}],
        # Passive citation
        [{"LOWER": {"IN": ["is", "was", "are", "were"]}}, {"LOWER": {"IN": ["cited", "reported", "claimed", "noted"]}}],
    ]
}

In [ ]:
@Language.component("metadiscourse_detector")
def metadiscourse_detector(doc):
    """Improved metadiscourse detector with contextual filtering"""
    if not Doc.has_extension("metadiscourse_markers"):
        Doc.set_extension("metadiscourse_markers", default={})
    
    # Initialize results with all categories
    results = {category: [] for category in list(INTERACTIVE_MARKERS.keys()) + list(INTERACTIONAL_MARKERS.keys())}
    
    # 1. DETECT MULTI-WORD MARKERS WITH PHRASEMATCHER
    phrase_matcher = PhraseMatcher(doc.vocab, attr="LOWER")
    
    # Add multi-word patterns for each category
    for category, markers in {**INTERACTIVE_MARKERS, **INTERACTIONAL_MARKERS}.items():
        multi_word_markers = [marker for marker in markers if " " in marker]
        if multi_word_markers:
            patterns = [nlp.make_doc(marker.lower()) for marker in multi_word_markers]
            phrase_matcher.add(category, None, *patterns)
    
    # Find multi-word matches
    multi_word_matches = phrase_matcher(doc)
    for match_id, start, end in multi_word_matches:
        category = doc.vocab.strings[match_id]
        # Store the match with text and position
        results[category].append((doc[start:end].text, (start, end)))
    
    # 2. DETECT CONTEXT-SPECIFIC PATTERNS WITH MATCHER
    context_matcher = Matcher(doc.vocab)
    
    # Add contextual patterns for disambiguation
    for category, patterns in CONTEXT_PATTERNS.items():
        context_matcher.add(category, patterns)
    
    # Find matches with contextual rules
    context_matches = context_matcher(doc)
    for match_id, start, end in context_matches:
        category = doc.vocab.strings[match_id]
        results[category].append((doc[start:end].text, (start, end)))
    
    # 3. DETECT SINGLE-TOKEN MARKERS WITH POS AND DEPENDENCY FILTERING
    for token in doc:
        # Skip tokens that are already part of multi-word matches
        if any(start <= token.i < end for category in results for _, (start, end) in results[category]):
            continue
        
        # HEDGES - check context to avoid false positives
        if token.lower_ in {"perhaps", "possibly", "probably", "approximately", "generally", "usually"}:
            results["hedges"].append((token.text, (token.i, token.i+1)))
        
        elif token.lower_ in {"may", "might", "could", "would"} and token.tag_ == "MD":
            # Check it's functioning as a hedge, not expressing ability/permission
            if token.head.pos_ == "VERB" and not any(c.text.lower() in {"able", "allowed"} for c in token.head.children):
                results["hedges"].append((token.text, (token.i, token.i+1)))
        
        # BOOSTERS - check context
        elif token.lower_ in {"clearly", "obviously", "certainly", "definitely", "always", "never"}:
            results["boosters"].append((token.text, (token.i, token.i+1)))
        
        # TRANSITIONS - check context to avoid false positives
        elif token.lower_ in {"however", "nevertheless", "therefore", "thus", "moreover", "furthermore"}:
            # More likely to be discourse function if at sentence start or after punctuation
            if token.is_sent_start or (token.i > 0 and doc[token.i-1].is_punct):
                results["transition_markers"].append((token.text, (token.i, token.i+1)))
        
        # SELF-MENTION - straightforward to identify
        elif token.lower_ in {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves"}:
            results["self_mention"].append((token.text, (token.i, token.i+1)))
        
        # ENGAGEMENT MARKERS - reader references
        elif token.lower_ in {"you", "your", "yours", "yourself", "yourselves"}:
            results["engagement_markers"].append((token.text, (token.i, token.i+1)))
        # 4. APPLY DEPENDENCY PARSING FOR COMPLEX CASES
    # Evidentials with reporting verbs
    for token in doc:
        # Reporting verbs with clausal complements indicate citation
        if token.lemma_.lower() in {"state", "argue", "claim", "report", "find", "note", "suggest"} and any(
            child.dep_ == "ccomp" or child.dep_ == "xcomp" for child in token.children):
            # Check if not already captured
            if not any(start <= token.i < end for _, (start, end) in results["evidentials"]):
                results["evidentials"].append((token.text, (token.i, token.i+1)))
    
    # Attitude markers with evaluative expressions
    for token in doc:
        if token.lemma_.lower() in {"agree", "disagree", "prefer"} and any(
            child.lower_ in {"i", "we"} for child in token.children):
            results["attitude_markers"].append((token.text, (token.i, token.i+1)))
    
    # Imperatives as engagement markers
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB" and not any(
            child.dep_ in {"nsubj", "csubj"} for child in token.children):
            # Initial verb with no subject - likely imperative 
            results["engagement_markers"].append((token.text, (token.i, token.i+1)))
    
    # Convert results to the format expected by the extension
    doc._.metadiscourse_markers = {
        category: [(text, span[0]) for text, span in markers] 
        for category, markers in results.items()
    }
    
    return doc

def process_corpus(corpus_data, nlp, batch_size=32):  # Reduced batch size for transformer model
    """Process corpus data with spaCy pipeline and extract metadiscourse statistics"""
    results = []
    total = len(corpus_data)
    
    print(f"Processing {total} documents...")
    
    # Process texts in batches with progress updates
    for i, (doc, meta_dict) in enumerate(nlp.pipe(corpus_data, as_tuples=True, batch_size=batch_size)):
        if i % 10 == 0 or i == total - 1:
            print(f"Processing document {i+1}/{total} ({(i+1)/total*100:.1f}%)...")
        
        # Extract base stats
        stats = {
            "Native_Language": meta_dict.get("l1", meta_dict.get("Native_Language", "unknown")),
            "text_id": meta_dict.get("text_id", str(i)),
            "word_count": len([t for t in doc if not t.is_punct and not t.is_space]),
            "sentence_count": len(list(doc.sents)),
        }
        
        # Extract counts for each metadiscourse category
        for category, markers in doc._.metadiscourse_markers.items():
            stats[f"{category}_count"] = len(markers)
            
            # Store actual markers for qualitative analysis
            if len(markers) > 0:
                stats[f"{category}_examples"] = "; ".join([m[0] for m in markers[:5]])
        
        # Copy other metadata fields
        for key, value in meta_dict.items():
            if key not in stats:
                stats[key] = value
        
        # Calculate normalized frequencies per 1000 words
        word_count = stats["word_count"]
        if word_count > 0:  # Avoid division by zero
            for key in list(stats.keys()):
                if key.endswith("_count") and key != "word_count" and key != "sentence_count":
                    stats[f"{key}_norm"] = stats[key] / word_count * 1000
        
        # Group into interactive vs. interactional
        interactive_count = sum(stats.get(f"{cat}_count", 0) for cat in INTERACTIVE_MARKERS.keys())
        interactional_count = sum(stats.get(f"{cat}_count", 0) for cat in INTERACTIONAL_MARKERS.keys())
        
        stats["interactive_count"] = interactive_count
        stats["interactional_count"] = interactional_count
        stats["total_metadiscourse_count"] = interactive_count + interactional_count
        
        if word_count > 0:
            stats["interactive_norm"] = interactive_count / word_count * 1000
            stats["interactional_norm"] = interactional_count / word_count * 1000
            stats["total_metadiscourse_norm"] = stats["interactive_norm"] + stats["interactional_norm"]
        
        results.append(stats)
    
    return pd.DataFrame(results)

In [ ]:
def analyze_metadiscourse_patterns(results_df, output_dir="."):
    """Analyze and visualize metadiscourse patterns in the corpus"""
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print("Generating analysis and visualizations...")
    
    # Ensure we have Native_Language column
    native_lang_col = "Native_Language"
    if native_lang_col not in results_df.columns:
        if "l1" in results_df.columns:
            native_lang_col = "l1"
            results_df["Native_Language"] = results_df["l1"]
        else:
            print(f"Warning: '{native_lang_col}' column not found. Creating dummy column.")
            results_df["Native_Language"] = "unknown"
    
    # Check if we have sufficient data for analysis
    if len(results_df) < 2:
        print("Warning: Not enough data for meaningful analysis.")
        return None, None
    
    # 1. OVERALL METADISCOURSE STATISTICS
    
    # Summary statistics for metadiscourse categories
    category_cols = [col for col in results_df.columns if col.endswith('_norm') and not col.endswith('total_norm')]
    summary_stats = results_df[category_cols].describe()
    
    # Group by Native Language
    if len(results_df[native_lang_col].unique()) > 1:
        lang_analysis = results_df.groupby(native_lang_col).agg({
            'interactive_norm': ['mean', 'std', 'count'],
            'interactional_norm': ['mean', 'std', 'count'],
            'total_metadiscourse_norm': ['mean', 'std', 'count']
        })
        
        # Category analysis by Native Language
        category_means = results_df.groupby(native_lang_col)[category_cols].mean()
        category_stds = results_df.groupby(native_lang_col)[category_cols].std()
    else:
        lang_analysis = None
        category_means = None
        print("Warning: Only one language group found, skipping language-based analysis.")
    
    # 2. VISUALIZATIONS
    
    try:
        # 1. Interactive vs. Interactional overall
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=results_df[['interactive_norm', 'interactional_norm']])
        plt.title('Distribution of Interactive vs. Interactional Metadiscourse')
        plt.ylabel('Frequency per 1000 words')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_type_distribution.png'))
        
        # 2. Metadiscourse categories overall
        plt.figure(figsize=(12, 8))
        plot_data = results_df[[col for col in category_cols if col in results_df.columns]]
        # Reorder columns by mean value for better visualization
        plot_data = plot_data[plot_data.mean().sort_values(ascending=False).index]
        sns.boxplot(data=plot_data)
        plt.title('Distribution of Metadiscourse Categories')
        plt.ylabel('Frequency per 1000 words')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_categories_distribution.png'))
        
        # 3. By Native Language (if available)
        if lang_analysis is not None and len(results_df[native_lang_col].unique()) > 1:
            # Interactive vs. Interactional by Native Language
            plt.figure(figsize=(12, 6))
            lang_means = results_df.groupby(native_lang_col)[['interactive_norm', 'interactional_norm']].mean()
            lang_means.plot(kind='bar')
            plt.title('Interactive vs. Interactional Metadiscourse by Native Language')
            plt.ylabel('Frequency per 1000 words')
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'metadiscourse_type_by_language.png'))
            
            # Top 5 categories by Native Language
            plt.figure(figsize=(14, 8))
            # Get top 5 categories by overall mean
            top_categories = results_df[category_cols].mean().sort_values(ascending=False).head(5).index
            if category_means is not None and len(top_categories) > 0:
                top_cat_data = category_means[top_categories]
                top_cat_data.plot(kind='bar')
                plt.title('Top 5 Metadiscourse Categories by Native Language')
                plt.ylabel('Frequency per 1000 words')
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, 'top_metadiscourse_categories.png'))
            
            # Heatmap of all categories
            if category_means is not None and len(category_means.columns) > 1:
                plt.figure(figsize=(16, 10))
                sns.heatmap(category_means, annot=True, cmap='YlGnBu', fmt='.2f')
                plt.title('Metadiscourse Markers by Category and Native Language')
                plt.tight_layout()
                plt.savefig(os.path.join(output_dir, 'metadiscourse_heatmap.png'))
    
    except Exception as e:
        print(f"Warning: Error generating visualizations: {e}")
        
    # 3. SAVE RESULTS
    
    # Save the full results dataframe
    results_df.to_csv(os.path.join(output_dir, 'metadiscourse_analysis_results.csv'), index=False)
    
    # Save summary statistics
    summary_stats.to_csv(os.path.join(output_dir, 'metadiscourse_summary_stats.csv'))
    
    # Save language analysis if available
    if lang_analysis is not None:
        lang_analysis.to_csv(os.path.join(output_dir, 'metadiscourse_by_language.csv'))
    
    # Save category means if available
    if category_means is not None:
        category_means.to_csv(os.path.join(output_dir, 'metadiscourse_category_means.csv'))
    
    print(f"Analysis complete. Results saved to {output_dir}/")
    
    return lang_analysis, category_means

In [ ]:
def calculate_shannon_entropy(frequencies):
    """
    Calculate Shannon entropy for a list of frequencies
    Higher values indicate more even distribution across texts
    """
    # Convert frequencies to probabilities
    total = sum(frequencies)
    if total == 0:
        return 0
    
    probabilities = [freq / total for freq in frequencies if freq > 0]
    
    # Calculate entropy
    entropy = -sum(p * np.log2(p) for p in probabilities)
    return entropy

def analyze_distribution(results_df, group_col="Native_Language", output_dir="."):
    """
    Analyze the distribution of metadiscourse markers across texts
    to identify potential skewness and imbalances
    """
    print("Analyzing distribution patterns across texts...")
    
    # Get metadiscourse category columns
    category_cols = [col for col in results_df.columns if col.endswith('_norm') 
                    and not col.endswith('total_norm')]
    
    # Initialize results dictionary and DataFrame for entropy values
    entropy_data = []
    distribution_stats = {}
    
    # For each language group
    for lang, group_df in results_df.groupby(group_col):
        distribution_stats[lang] = {}
        
        # For each metadiscourse category
        for category in category_cols:
            # Get non-zero values (texts that actually use this marker)
            values = group_df[category].values
            non_zero_values = values[values > 0]
            
            # Calculate distribution metrics
            entropy = calculate_shannon_entropy(values)
            cv = np.std(values) / np.mean(values) if len(values) > 0 and np.mean(values) > 0 else 0
            
            # Calculate Gini coefficient (measure of inequality)
            if len(values) > 0 and np.mean(values) > 0:
                # Sort values
                sorted_values = np.sort(values)
                # Calculate cumulative sum
                cumsum = np.cumsum(sorted_values)
                # Calculate Lorenz curve
                lorenz_curve = cumsum / cumsum[-1]
                # Normalize x-axis
                x = np.linspace(0, 1, len(lorenz_curve))
                # Calculate area under Lorenz curve
                area_under_lorenz = np.trapz(lorenz_curve, x)
                # Gini coefficient = 1 - 2 * area under Lorenz curve
                gini = 1 - 2 * area_under_lorenz
            else:
                gini = 0
            
            # Store metrics
            distribution_stats[lang][category] = {
                'entropy': entropy,
                'cv': cv,  # Coefficient of variation (higher values = more skewed)
                'gini': gini,  # Gini coefficient (higher values = more inequality)
                'mean': np.mean(values),
                'median': np.median(values),
                'zeros_pct': np.sum(values == 0) / len(values) * 100,  # % of texts not using this marker
                'count_non_zero': len(non_zero_values),
                'total_texts': len(group_df)
            }
            
            # Add to entropy DataFrame for visualization
            entropy_data.append({
                'Language': lang,
                'Category': category.replace('_norm', ''),
                'Entropy': entropy,
                'CV': cv,
                'Gini': gini
            })
    
    # Convert to DataFrame for easier analysis
    entropy_df = pd.DataFrame(entropy_data)
    
    # Visualize entropy distribution
    try:
        plt.figure(figsize=(14, 8))
        entropy_pivot = entropy_df.pivot(index='Category', columns='Language', values='Entropy')
        sns.heatmap(entropy_pivot, annot=True, cmap='YlGnBu', fmt='.2f')
        plt.title('Shannon Entropy by Category and Language')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_entropy_heatmap.png'))
        
        # Create distribution plot for top categories
        top_categories = [cat.replace('_norm', '') for cat in results_df[category_cols].mean().sort_values(ascending=False).head(5).index]
        
        fig, axes = plt.subplots(len(top_categories), 1, figsize=(12, 4*len(top_categories)))
        
        for i, category in enumerate(top_categories):
            ax = axes[i] if len(top_categories) > 1 else axes
            for lang, group_df in results_df.groupby(group_col):
                sns.kdeplot(group_df[f"{category}_norm"], label=lang, ax=ax)
            
            ax.set_title(f'Distribution of {category}')
            ax.set_xlabel('Frequency per 1000 words')
            ax.legend()
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_distribution_density.png'))
    
    except Exception as e:
        print(f"Warning: Error generating distribution visualizations: {e}")
    
    # Save distribution statistics
    with open(os.path.join(output_dir, 'distribution_statistics.json'), 'w') as f:
        # Convert numpy types to Python types for JSON serialization
        import json
        def convert_to_serializable(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj
        
        serializable_stats = {lang: {cat: {k: convert_to_serializable(v) 
                                          for k, v in metrics.items()} 
                                    for cat, metrics in cat_stats.items()} 
                             for lang, cat_stats in distribution_stats.items()}
        
        json.dump(serializable_stats, f, indent=2)
    
    return entropy_df, distribution_stats

In [ ]:
def detect_and_handle_outliers(results_df, category_cols, z_score_threshold=3.0):
    """
    Detect outliers in metadiscourse marker usage and create a filtered DataFrame
    
    Returns:
    - outlier_df: DataFrame with outlier flags
    - filtered_df: DataFrame with outliers removed or winsorized
    """
    outlier_df = pd.DataFrame(index=results_df.index)
    filtered_df = results_df.copy()
    
    print(f"Detecting outliers using z-score threshold of {z_score_threshold}...")
    
    # For each metadiscourse category
    for category in category_cols:
        # Calculate z-scores within each language group
        for lang, group_df in results_df.groupby('Native_Language'):
            indices = group_df.index
            values = group_df[category].values
            
            if len(values) > 5:  # Only apply to groups with sufficient data
                mean, std = np.mean(values), np.std(values)
                if std > 0:
                    z_scores = (values - mean) / std
                    
                    # Flag outliers
                    outliers = np.abs(z_scores) > z_score_threshold
                    outlier_df.loc[indices, f"{category}_outlier"] = outliers
                    
                    # Winsorize outliers (cap at threshold)
                    upper_bound = mean + z_score_threshold * std
                    lower_bound = mean - z_score_threshold * std
                    
                    filtered_values = np.clip(values, lower_bound, upper_bound)
                    filtered_df.loc[indices, category] = filtered_values
    
    # Count outliers per document
    outlier_df['total_outliers'] = outlier_df.sum(axis=1)
    
    print(f"Detected {outlier_df['total_outliers'].sum()} outlier values across {(outlier_df['total_outliers'] > 0).sum()} documents")
    
    return outlier_df, filtered_df

def normalize_and_compare(results_df, group_col="Native_Language", normalize_method="log", output_dir="."):
    """
    Perform comparison between language groups with normalized values
    to account for different distributions
    
    normalize_method: 'log', 'rank', 'z-score', or 'proportion'
    """
    print(f"Performing normalized comparison using {normalize_method} normalization...")
    
    # Get metadiscourse category columns
    category_cols = [col for col in results_df.columns if col.endswith('_norm') 
                    and not col.endswith('total_norm')]
    
    # Create a copy for normalized values
    normalized_df = results_df.copy()
    
    # Apply normalization
    if normalize_method == "log":
        # Log normalization (add 1 to avoid log(0))
        for col in category_cols:
            normalized_df[f"{col}_normalized"] = np.log1p(normalized_df[col])
    
    elif normalize_method == "rank":
        # Rank normalization (within each language group)
        for lang, group_df in normalized_df.groupby(group_col):
            for col in category_cols:
                normalized_df.loc[group_df.index, f"{col}_normalized"] = group_df[col].rank(pct=True)
    
    elif normalize_method == "z-score":
        # Z-score normalization (within each language group)
        for lang, group_df in normalized_df.groupby(group_col):
            for col in category_cols:
                mean = group_df[col].mean()
                std = group_df[col].std()
                if std > 0:
                    normalized_df.loc[group_df.index, f"{col}_normalized"] = (group_df[col] - mean) / std
    
    elif normalize_method == "proportion":
        # Express as proportion of total metadiscourse
        for idx, row in normalized_df.iterrows():
            total = row['total_metadiscourse_norm']
            if total > 0:
                for col in category_cols:
                    normalized_df.loc[idx, f"{col}_normalized"] = row[col] / total
    
    # Group by language and calculate means of normalized values
    norm_cols = [f"{col}_normalized" for col in category_cols]
    normalized_means = normalized_df.groupby(group_col)[norm_cols].mean()
    
    # Visualize normalized comparison
    try:
        # Create heatmap of normalized values
        plt.figure(figsize=(16, 10))
        # Remove the "_normalized" suffix for cleaner display
        display_means = normalized_means.copy()
        display_means.columns = [col.replace('_normalized', '') for col in display_means.columns]
        
        # Sort columns by mean value
        col_order = display_means.mean().sort_values(ascending=False).index
        display_means = display_means[col_order]
        
        sns.heatmap(display_means, annot=True, cmap='YlGnBu', fmt='.2f')
        plt.title(f'Normalized Metadiscourse Markers by Language ({normalize_method})')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'metadiscourse_normalized_{normalize_method}.png'))
    
    except Exception as e:
        print(f"Warning: Error generating normalized comparison visualizations: {e}")
    
    # Save normalized means
    normalized_means.to_csv(os.path.join(output_dir, f'metadiscourse_normalized_{normalize_method}.csv'))
    
    return normalized_df, normalized_means

In [ ]:
def extract_examples_from_results(results_df, output_dir="."):
    """Extract and save examples of each metadiscourse category for qualitative analysis"""
    example_file = os.path.join(output_dir, 'metadiscourse_examples.txt')
    
    with open(example_file, 'w') as f:
        f.write("METADISCOURSE MARKER EXAMPLES\n")
        f.write("============================\n\n")
        
        for category in list(INTERACTIVE_MARKERS.keys()) + list(INTERACTIONAL_MARKERS.keys()):
            f.write(f"{category.upper()}\n")
            f.write("-" * len(category) + "\n")
            
            # Get examples for this category
            example_col = f"{category}_examples"
            if example_col in results_df.columns:
                examples = set()
                for ex_str in results_df[example_col].dropna():
                    examples.update(ex_str.split('; '))
                
                for example in sorted(examples)[:20]:  # Limit to 20 examples per category
                    f.write(f"- {example}\n")
            
            f.write("\n")
    
    print(f"Examples saved to {example_file}")

def enhance_visualizations(results_df, output_dir="."):
    """
    Add enhanced visualizations to complement the existing analysis
    """
    print("Generating additional visualizations...")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Ensure we have Native_Language column
    native_lang_col = "Native_Language"
    if native_lang_col not in results_df.columns:
        if "l1" in results_df.columns:
            native_lang_col = "l1"
            results_df["Native_Language"] = results_df["l1"]
        else:
            print(f"Warning: '{native_lang_col}' column not found. Using dummy value.")
            results_df["Native_Language"] = "unknown"
    
    # Get metadiscourse category columns
    category_cols = [col for col in results_df.columns if col.endswith('_norm') 
                    and not col.endswith('total_norm')]
    
    try:
        # 1. Interactive vs Interactional stacked bar chart by language
        plt.figure(figsize=(14, 8))
        lang_means = results_df.groupby(native_lang_col)[['interactive_norm', 'interactional_norm']].mean()
        lang_means.plot(kind='bar', stacked=True, colormap='viridis')
        plt.title('Interactive vs. Interactional Metadiscourse by Native Language')
        plt.ylabel('Frequency per 1000 words')
        plt.xlabel('Native Language')
        plt.legend(title='Type')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_stacked_by_language.png'))
        print(f"✓ Created stacked bar chart: metadiscourse_stacked_by_language.png")
        
        # 2. Top 5 categories - horizontal bar chart by language
        plt.figure(figsize=(14, 10))
        # Get top 5 categories by overall mean
        top_categories = results_df[category_cols].mean().sort_values(ascending=False).head(5).index
        
        # Create DataFrame for plotting
        plot_data = []
        for lang, group_df in results_df.groupby(native_lang_col):
            for cat in top_categories:
                plot_data.append({
                    'Language': lang,
                    'Category': cat.replace('_norm', ''),
                    'Frequency': group_df[cat].mean()
                })
        
        plot_df = pd.DataFrame(plot_data)
        
        # Create the plot
        sns.barplot(x='Frequency', y='Category', hue='Language', data=plot_df)
        plt.title('Top 5 Metadiscourse Categories by Native Language')
        plt.xlabel('Frequency per 1000 words')
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'top_categories_horizontal.png'))
        print(f"✓ Created horizontal bar chart: top_categories_horizontal.png")
        
        # 3. Metadiscourse profile spider plot
        if len(results_df[native_lang_col].unique()) > 1:
            category_means = results_df.groupby(native_lang_col)[category_cols].mean()
            # Select top 8 categories for readability
            top_8_categories = results_df[category_cols].mean().sort_values(ascending=False).head(8).index
            cat_means_subset = category_means[top_8_categories]
            
            # Create radar chart
            fig = plt.figure(figsize=(14, 12))
            ax = fig.add_subplot(111, polar=True)
            
            # Set number of categories and angles
            categories = [cat.replace('_norm', '') for cat in cat_means_subset.columns]
            N = len(categories)
            angles = [n / float(N) * 2 * np.pi for n in range(N)]
            angles += angles[:1]  # Close the loop
            
            # Plot each language group
            for i, (lang, values) in enumerate(cat_means_subset.iterrows()):
                values_list = values.tolist()
                values_list += values_list[:1]  # Close the loop
                
                ax.plot(angles, values_list, linewidth=2, linestyle='solid', label=lang)
                ax.fill(angles, values_list, alpha=0.1)
            
            # Set category labels
            plt.xticks(angles[:-1], categories, size=12)
            
            # Add legend and title
            plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
            plt.title('Metadiscourse Profile by Native Language', size=15, y=1.1)
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'metadiscourse_radar_improved.png'))
            print(f"✓ Created radar chart: metadiscourse_radar_improved.png")
        
        # 4. Violin plots to show distribution
        plt.figure(figsize=(16, 10))
        
        # Reshape data for violin plot
        plot_data = []
        for lang, group_df in results_df.groupby(native_lang_col):
            for cat in top_categories:
                for val in group_df[cat].values:
                    plot_data.append({
                        'Language': lang,
                        'Category': cat.replace('_norm', ''),
                        'Frequency': val
                    })
        
        plot_df = pd.DataFrame(plot_data)
        
        # Create violin plot
        sns.violinplot(x='Category', y='Frequency', hue='Language', data=plot_df,
                      split=True, inner="quart", linewidth=1)
        plt.title('Distribution of Top Metadiscourse Categories by Native Language')
        plt.ylabel('Frequency per 1000 words')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_violin_plot.png'))
        print(f"✓ Created violin plot: metadiscourse_violin_plot.png")
        
        # 5. Metadiscourse usage correlation heatmap
        plt.figure(figsize=(14, 12))
        correlation = results_df[category_cols].corr()
        mask = np.triu(correlation)
        sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', 
                   mask=mask, vmin=-1, vmax=1)
        plt.title('Correlation Between Metadiscourse Categories')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'metadiscourse_correlation.png'))
        print(f"✓ Created correlation heatmap: metadiscourse_correlation.png")
        
        return True
        
    except Exception as e:
        print(f"Error generating visualizations: {e}")
        import traceback
        traceback.print_exc()
        return False

In [ ]:
def run_analysis(corpus_path=None, output_dir="metadiscourse_results", extract_examples=True,
                 outlier_threshold=3.0, normalization_method="log"):
    """
    Run the metadiscourse analysis with distribution analysis and normalization
    
    Parameters:
    -----------
    corpus_path : str, optional
        Path to the CSV file with texts and metadata. If None, uses the already loaded data.
    output_dir : str
        Directory to save output files
    extract_examples : bool
        Whether to extract examples of each metadiscourse category
    outlier_threshold : float
        Z-score threshold for outlier detection
    normalization_method : str
        Method for normalization: 'log', 'rank', 'z-score', or 'proportion'
    """
    # Set up spaCy pipeline
    print("Loading spaCy transformer model...")
    global nlp
    
    # Check if metadiscourse_detector is already in the pipeline
    if "metadiscourse_detector" not in nlp.pipe_names:
        nlp.add_pipe("metadiscourse_detector", last=True)
        print("Added metadiscourse_detector to pipeline")
    
    if corpus_path:
        # Load data if path is provided
        print(f"Loading data from {corpus_path}...")
        try:
            meta = pd.read_csv(corpus_path)
            text = pd.read_csv(corpus_path)
            
            print("Preprocessing text...")
            pattern = r'[^\w\s.,?!]'
            pattern_ = r'\s+'
            text['text_field'] = text['text_field'].apply(lambda x: re.sub(pattern, '', str(x)).replace('\n', ''))
            text['text_field'] = text['text_field'].apply(lambda x: re.sub(pattern_, ' ', x))
            text['text_field'] = text['text_field'].str.lower()
            
            # Create corpus data format
            meta_x = meta.to_dict('records')
            text_only = text['text_field'].values.tolist()
            corpus_data = list(zip(text_only, meta_x))
        except Exception as e:
            print(f"Error loading corpus: {e}")
            return None
    else:
        # Use existing loaded data (from user's code snippet)
        print("Using already loaded data")
        corpus_data = icle
    
    print(f"Processing {len(corpus_data)} documents for metadiscourse markers...")
    results_df = process_corpus(corpus_data, nlp)
    
    print("Analyzing metadiscourse patterns...")
    lang_analysis, category_means = analyze_metadiscourse_patterns(results_df, output_dir)
    
    # Get category columns for further analysis
    category_cols = [col for col in results_df.columns if col.endswith('_norm') 
                    and not col.endswith('total_norm')]
    
    # Analyze distribution across texts
    print("Analyzing distribution evenness...")
    entropy_df, distribution_stats = analyze_distribution(results_df, "Native_Language", output_dir)
    
    # Detect and handle outliers
    print("Detecting and handling outliers...")
    outlier_df, filtered_df = detect_and_handle_outliers(results_df, category_cols, outlier_threshold)
    outlier_df.to_csv(os.path.join(output_dir, 'metadiscourse_outliers.csv'))
    filtered_df.to_csv(os.path.join(output_dir, 'metadiscourse_filtered.csv'))
    
    # Perform normalized comparison
    print(f"Performing normalized comparison using {normalization_method}...")
    normalized_df, normalized_means = normalize_and_compare(
        filtered_df, "Native_Language", normalization_method, output_dir)
    
    # Generate enhanced visualizations
    print("\nGenerating enhanced visualizations...")
    enhance_visualizations(results_df, output_dir)
    
    # Extract examples if requested
    if extract_examples and results_df is not None:
        extract_examples_from_results(results_df, output_dir)
    
    # Display summary statistics
    if results_df is not None:
        print("\n--- Metadiscourse Analysis Summary ---")
        print(f"\nTotal documents analyzed: {len(results_df)}")
        
        print("\nOverall metadiscourse frequencies (per 1000 words):")
        print(f"  Interactive: {results_df['interactive_norm'].mean():.2f}")
        print(f"  Interactional: {results_df['interactional_norm'].mean():.2f}")
        print(f"  Total: {results_df['total_metadiscourse_norm'].mean():.2f}")
        
        print("\nTop 5 metadiscourse categories:")
        top_cats = results_df[category_cols].mean().sort_values(ascending=False).head(5)
        for cat, value in top_cats.items():
            print(f"  {cat.replace('_norm', '')}: {value:.2f}")
        
        print("\nDistribution evenness (Shannon entropy):")
        for lang, values in distribution_stats.items():
            print(f"\n  {lang}:")
            # Get top 3 most and least evenly distributed categories
            entropies = [(cat, metrics['entropy']) for cat, metrics in values.items()]
            entropies.sort(key=lambda x: x[1], reverse=True)
            
            print("    Most evenly distributed:")
            for cat, entropy in entropies[:3]:
                print(f"      {cat.replace('_norm', '')}: {entropy:.2f}")
            
            print("    Least evenly distributed:")
            for cat, entropy in entropies[-3:]:
                print(f"      {cat.replace('_norm', '')}: {entropy:.2f}")
    
    # Return all analytical results
    results = {
        'raw_results': results_df,
        'language_analysis': lang_analysis,
        'category_means': category_means,
        'entropy': entropy_df,
        'distribution_stats': distribution_stats,
        'outliers': outlier_df,
        'filtered_results': filtered_df,
        'normalized_results': normalized_df,
        'normalized_means': normalized_means
    }
    
    return results


# Run the analysis
results = run_analysis(outlier_threshold=3.0, normalization_method="log")